In [1]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from utils import find_nearest_gene_atac_peaks, preprocessing_rna, preprocessing_atac, preprocessing_prot
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from tqdm import tqdm
tqdm.pandas()

Dataset has been downloaded from:
https://zenodo.org/records/7041849

In [2]:
rna = sc.read_h5ad("./data/FrangiehIzar2021_RNA.h5ad")
rna

AnnData object with n_obs × n_vars = 218331 × 23712
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'umap_x', 'umap_y', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensembl_id', 'ncounts', 'ncells'

In [3]:
prot = sc.read_h5ad("./data/FrangiehIzar2021_protein.h5ad")
prot

AnnData object with n_obs × n_vars = 218331 × 24
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'umap_x', 'umap_y', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'Target', 'Clone', 'TotalSeq A#', 'Barcode', 'Isotype_control', 'ncounts', 'ncells'

In [4]:
random_state = np.random.RandomState()
indices = random_state.permutation(list(range(len(rna.obs))))
rna = rna[indices,:]
prot = prot[indices,:]

# RNA layer preprocessing

In [5]:
rna = rna.copy()

In [6]:
rna.X = np.array(rna.X.todense())

In [7]:
rna.var["gene_stable_id"] = rna.var["ensembl_id"]

In [8]:
rna = preprocessing_rna(rna)
rna

AnnData object with n_obs × n_vars = 218331 × 23711
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'umap_x', 'umap_y', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'ensembl_id', 'ncounts', 'ncells', 'gene_stable_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

# Prot layer preprocesssing

In [9]:
prot.X = np.array(prot.X.todense())

In [10]:
prot.var["gene_stable_id"] = [
    "ENSG00000157404",
    "ENSG00000027697",
    "ENSG00000134853",
    "ENSG00000113721",
    "ENSG00000198053", 
    "ENSG00000121966",
    "ENSG00000120156", 
    "ENSG00000120217",
    "ENSG00000150093",
    "ENSG00000128052",
    "ENSG00000026508",
    "ENSG00000196776",
    "ENSG00000091409",
    "ENSG00000116815",
    "ENSG00000085063",
    "ENSG00000259207",
    "ENSG00000206503",
    np.nan,
    np.nan,
    np.nan,
    np.nan,
    "ENSG00000204592",
    "ENSG00000010278",
    "ENSG00000188389",
]

In [11]:
prot = prot[:, ~prot.var["gene_stable_id"].isnull()]
prot

View of AnnData object with n_obs × n_vars = 218331 × 20
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'umap_x', 'umap_y', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'Target', 'Clone', 'TotalSeq A#', 'Barcode', 'Isotype_control', 'ncounts', 'ncells', 'gene_stable_id'

In [12]:
prot = preprocessing_prot(prot, k=10)
prot

AnnData object with n_obs × n_vars = 218331 × 20
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'umap_x', 'umap_y', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'Target', 'Clone', 'TotalSeq A#', 'Barcode', 'Isotype_control', 'ncounts', 'ncells', 'gene_stable_id'
    uns: 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

# Create MuData and save

In [13]:
mdata = mu.MuData({"rna": rna, 
                   "prot": prot})

In [14]:
mu.pp.intersect_obs(mdata)

In [15]:
mdata.write("./data/Frangieh2021.h5mu")